# Leakage

Data from an experiment design built for a two-level system can be used to fit a *three*-level model, which is how you detect and quantify leakage out of the computational subspace. This page shows how, and how to generate a report whose gate error metrics respect the distinguished role of the first two levels.

## Why a third level

A qubit is usually the lowest two levels of something larger, such as a transmon's anharmonic ladder. When a control pulse has spectral weight on a neighboring transition, population leaves the computational subspace. That is leakage.

Leakage hides information rather than destroying it. Population that leaves can come back, carrying a phase that depends on how long it was gone. A two-level model has nowhere to put that population, hence nowhere to put the memory, and the data ends up matching no single Markovian two-level gate set. Fit three levels and the memory becomes explicit: the leaked amplitude has somewhere to live, and the leaky gate is once again a fixed CPTP map applied identically at every occurrence. What looks non-Markovian in two levels is ordinary Markovian error in three.

Note that this design's fiducials and germs were chosen to amplify two-level parameters, with no guarantee that they amplify every three-level one.

In [1]:
from pygsti.modelpacks import smq1Q_XYI as mp
from pygsti.leakage import leaky_qubit_model_from_pspec, construct_leakage_report
from pygsti.data import simulate_data
from pygsti.protocols import StandardGST, ProtocolData
import numpy as np
import scipy.linalg as la

## A coherent leakage error

`with_leaky_gate` draws a real unit vector $v$ whose first component is zero, forms the rank-one generator $H = s\,vv^{T}$, and composes $U = \exp(iH)$ after the ideal gate. The zero first component is what makes this a leakage error rather than a generic qutrit error: it confines $v$ to levels 1 and 2, so the ground state is untouched and amplitude moves only between the excited computational level and the leakage level. That is the picture for a weakly anharmonic transmon, whose 0-1 drive is detuned from 1-2 by only the anharmonicity. At `strength=0.125` about 0.4% of the population in level 1 leaks per application.

On three levels the error is a unitary: exactly CPTP, exactly Markovian, the same at every occurrence. Restricted to the computational subspace it is not even trace preserving.

In [2]:
def with_leaky_gate(m, gate_label, strength):
    rng = np.random.default_rng(0)
    v = np.concatenate([[0.0], rng.standard_normal(size=(2,))])
    v /= la.norm(v)
    H = v.reshape((-1, 1)) @ v.reshape((1, -1))
    H *= strength
    U = la.expm(1j*H)
    m_copy = m.copy()
    G_ideal = m_copy.operations[gate_label]
    from pygsti.modelmembers.operations import ComposedOp, StaticUnitaryOp
    m_copy.operations[gate_label] = ComposedOp([G_ideal, StaticUnitaryOp(U, basis=m.basis)])
    return m_copy, v

## The target model

`leaky_qubit_model_from_pspec` returns a qutrit *lift* of the qubit model, not a leaky one; the name is a misnomer. Each 2-by-2 gate unitary $u$ is promoted to the 3-by-3 unitary with $u$ in the upper-left block and a 1 in the remaining diagonal entry, so the ideal gates act trivially on the third level. The preparation is $|0\rangle\langle 0|$, and `Mdefault` keeps exactly two effects: $|0\rangle\langle 0|$ for outcome "0", and $|1\rangle\langle 1| + |2\rangle\langle 2|$ for outcome "1". The leakage level is not separately observable; it reads out as a 1. That is why the two-level design's data can be used unchanged.

The default `mx_basis='l2p1'` ("leakage, 2 plus 1") is a Hermitian qutrit basis sorted by that split: four elements span the computational subspace's operators (the qubit identity and Paulis, padded with a zero row and column), one is the projector onto the leakage level, and four span the coherences between them. Hermiticity is required because these models hold real parameter vectors; the `C[...]`/`L[...]` labels matter as much, since pyGSTi reads them to decide that a basis designates a *proper* computational subspace, which switches on the subspace-restricted gauge optimization and report metrics below.

In [3]:
ed = mp.create_gst_experiment_design(max_max_length=8)
tm3 = leaky_qubit_model_from_pspec(mp.processor_spec(), mx_basis='l2p1')
dgm3, leaking_state = with_leaky_gate(tm3, ('Gxpi2', 0), strength=0.125)

## Simulating and fitting

Short circuits carry less information, so `num_samples` is large: $10^5$ shots buy back some of the precision the missing depth would have supplied. That count also outruns pyGSTi's default likelihood regularization: `min_prob_clip` and `radius` both default to $10^{-4}$, while the smallest expected frequency here is $10^{-5}$. The cell below lowers both to $10^{-12}$, which keeps the regularization out of the region where the fit is decided but leaves the objective harder on the optimizer — the fit hits its iteration cap and warns about it at every stage. [Judging fits](JudgingTheFit) explains the tradeoff.

In [4]:
num_samples = 100_000
if num_samples > 10_000:
    from pygsti.objectivefns import objectivefns
    objectivefns.DEFAULT_MIN_PROB_CLIP = objectivefns.DEFAULT_RADIUS = 1e-12
ds = simulate_data(dgm3, ed.all_circuits_needing_data, num_samples=num_samples, seed=1997)

The bad-fit machinery acts only when the misfit exceeds `threshold` standard deviations, so setting it to 0.0 makes the `wildcard1d` analysis run for essentially any fit. A wildcard budget is the slack you would have to allow the model's predicted probabilities to make them consistent with the data. The one-dimensional version fits a single scale, spread across gates in proportion to each gate's diamond distance from its target, and the report prints it beside the other error metrics.

In [5]:
gst = StandardGST(
    modes=('CPTPLND',), target_model=tm3, verbosity=4,
    badfit_options={'actions': ['wildcard1d'], 'threshold': 0.0}
)
pd = ProtocolData(ed, ds)
res = gst.run(pd)

-- Std Practice:  Iter 1 of 1  (CPTPLND) --: 
    Precomputing CircuitOutcomeProbabilityArray layouts for each iteration.
      Using MapForwardSimulator without MPI
      Using MapForwardSimulator without MPI
      Using MapForwardSimulator without MPI
      Using MapForwardSimulator without MPI
  --- Iterative GST: Iter 1 of 4  92 circuits ---: 
    --- chi2 GST ---
      --- Outer Iter 0: norm_f = 3.27659e+06, mu=1, |x|=9.1849e-14, |J|=10983.8
      --- Outer Iter 1: norm_f = 2.25506e+06, mu=15206.4, |x|=0.0320463, |J|=391545
      --- Outer Iter 2: norm_f = 196681, mu=15206.4, |x|=0.0759232, |J|=29102
      --- Outer Iter 3: norm_f = 9358.6, mu=5068.79, |x|=0.137324, |J|=12109.3
      --- Outer Iter 4: norm_f = 2714.04, mu=4647.75, |x|=0.105592, |J|=12113.7
      --- Outer Iter 5: norm_f = 555.922, mu=3527.47, |x|=0.0926559, |J|=12408.1
      --- Outer Iter 6: norm_f = 210.271, mu=3379.13, |x|=0.0880471, |J|=12685.6
      --- Outer Iter 7: norm_f = 176.118, mu=4061.15, |x|=0.088443

      --- Outer Iter 0: norm_f = 157.354, mu=1, |x|=0.0900809, |J|=18167.8
      --- Outer Iter 1: norm_f = 121.629, mu=30938.1, |x|=0.0904584, |J|=18077.5
      --- Outer Iter 2: norm_f = 118.572, mu=12213.1, |x|=0.091333, |J|=18071.9
      --- Outer Iter 3: norm_f = 117.713, mu=9811.59, |x|=0.093028, |J|=18084.1
      --- Outer Iter 4: norm_f = 117.449, mu=21214.3, |x|=0.0941274, |J|=18091
      --- Outer Iter 5: norm_f = 117.253, mu=28766.1, |x|=0.0951735, |J|=18086.4
      --- Outer Iter 6: norm_f = 117.099, mu=81640.4, |x|=0.0955296, |J|=18091.6
      --- Outer Iter 7: norm_f = 116.545, mu=81638.5, |x|=0.0957773, |J|=18086
      --- Outer Iter 8: norm_f = 116.38, mu=83185, |x|=0.0960255, |J|=18086.5
      --- Outer Iter 9: norm_f = 116.247, mu=84587.7, |x|=0.0962669, |J|=18085
      --- Outer Iter 10: norm_f = 116.203, mu=110694, |x|=0.0965055, |J|=18085.7
      --- Outer Iter 11: norm_f = 116.084, mu=110707, |x|=0.0966847, |J|=18084.1
      --- Outer Iter 12: norm_f = 116.031, mu

      --- Outer Iter 0: norm_f = 270.408, mu=1, |x|=0.110098, |J|=26448.1
      --- Outer Iter 1: norm_f = 220.828, mu=64144.4, |x|=0.109504, |J|=26509.6
      --- Outer Iter 2: norm_f = 219.758, mu=77742.4, |x|=0.108839, |J|=26540.2
      --- Outer Iter 3: norm_f = 218.235, mu=79628.4, |x|=0.108524, |J|=26545
      --- Outer Iter 4: norm_f = 217.877, mu=110170, |x|=0.10831, |J|=26545.2
      --- Outer Iter 5: norm_f = 217.542, mu=153596, |x|=0.108202, |J|=26547.6
      --- Outer Iter 6: norm_f = 216.822, mu=155871, |x|=0.108124, |J|=26538.9
      --- Outer Iter 7: norm_f = 216.474, mu=312036, |x|=0.108085, |J|=26538.4
      --- Outer Iter 8: norm_f = 216.389, mu=313398, |x|=0.108048, |J|=26536
      --- Outer Iter 9: norm_f = 216.351, mu=315986, |x|=0.108016, |J|=26536.9
      --- Outer Iter 10: norm_f = 216.322, mu=315986, |x|=0.107986, |J|=26535.9
      --- Outer Iter 11: norm_f = 216.299, mu=313076, |x|=0.10796, |J|=26536.2
      --- Outer Iter 12: norm_f = 216.278, mu=285494, |x|=

  --- Iterative GST: Iter 4 of 4  448 circuits ---: 
    --- chi2 GST ---
      --- Outer Iter 0: norm_f = 413.197, mu=1, |x|=0.109395, |J|=41938.5
      --- Outer Iter 1: norm_f = 382.476, mu=227899, |x|=0.108697, |J|=41961.7
      --- Outer Iter 2: norm_f = 378.49, mu=209811, |x|=0.107774, |J|=41945.9
      --- Outer Iter 3: norm_f = 376.465, mu=210063, |x|=0.106956, |J|=42000.4
      --- Outer Iter 4: norm_f = 373.782, mu=210058, |x|=0.106279, |J|=41944.9
      --- Outer Iter 5: norm_f = 372.592, mu=230109, |x|=0.105697, |J|=41989.8
      --- Outer Iter 6: norm_f = 370.563, mu=231072, |x|=0.105235, |J|=41943.2
      --- Outer Iter 7: norm_f = 369.718, mu=269166, |x|=0.104818, |J|=41975.8
      --- Outer Iter 8: norm_f = 368.065, mu=269818, |x|=0.104496, |J|=41939.6
      --- Outer Iter 9: norm_f = 367.298, mu=285394, |x|=0.104192, |J|=41959.5
      --- Outer Iter 10: norm_f = 366.34, mu=287275, |x|=0.103928, |J|=41937
      --- Outer Iter 11: norm_f = 365.837, mu=311036, |x|=0.10367

      -- Performing 'stdgaugeopt' gauge optimization on CPTPLND estimate --
    ******************* Adding Wildcard Budget **************************
    Beginning wildcard budget optimization using alpha bisection method.
      Searching for interval [None, None] with guess 0.1
      Guess value is feasible, 
      Searching for interval [0.1, None] with guess 0.05
      Guess value is feasible, 
      Searching for interval [0.05, None] with guess 0.025
      Guess value is feasible, 
      Searching for interval [0.025, None] with guess 0.0125
      Guess value is feasible, 
      Searching for interval [0.0125, None] with guess 0.00625
      Guess value is feasible, 
      Searching for interval [0.00625, None] with guess 0.003125
      Guess value is feasible, 
      Searching for interval [0.003125, None] with guess 0.0015625
      Guess value is feasible, 
      Searching for interval [0.0015625, None] with guess 0.00078125
      Guess value is feasible, 
      Searching for int

## Reporting

Ordinary gauge optimization minimizes the distance between estimate and target over the gauge group, comparing full 9-by-9 superoperators. That is the wrong objective here: the target's action on the leakage level is a convention we chose, not anything the device was built to do. `construct_leakage_report` compares only the computational block, then finishes with a step restricted to the block-diagonal subgroup that does not mix the computational subspace with the leakage level.

Each estimate in `updated_res` gains a model keyed `LAGO` beside the usual `stdgaugeopt` one, and the two disagree usefully. The error was injected into `Gxpi2` alone; the leakage-aware gauge attributes it there far more sharply than standard gauge optimization does, which pushes more of it onto `Gypi2`. How large the gap looks depends on which fidelity measure you compare them with, but its direction does not. The report's gate tables also gain leakage columns and a switch between subspace-restricted and full-space metrics.

In [6]:
report_dir = '../../../example_files/leakage-report-automagic'
report_object, updated_res = construct_leakage_report(res, title='easy leakage analysis!')
report_object.write_html(report_dir, connected=True)

<repo>/pygsti/models/explicitmodel.py:634: UnknownGaugeSpaceDimension: ExplicOpModel.num_modeltest_params could not obtain number of *non-gauge* parameters - using total instead
  _warnings.warn(("ExplicOpModel.num_modeltest_params could not obtain number of *non-gauge* parameters"


Running idle tomography
Computing switchable properties


<repo>/pygsti/report/reportables.py:1064: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  prob.solve(solver=s)
<repo>/pygsti/report/reportables.py:1064: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  prob.solve(solver=s)
<repo>/pygsti/report/reportables.py:1064: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  prob.solve(solver=s)
<repo>/pygsti/report/reportables.py:1064: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  prob.solve(solver=s)
<repo>/pygsti/report/reportables.py:1064: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more inform

Served with these docs: <a href="../../../reports/leakage-report-automagic.html">leakage-report-automagic</a>.